# Gerando frases positivas a partir de todas as frases positivas reais.


In [5]:
# Compara o resultado da avalição dos modelos Bert.
# Manipulação de dados
import pandas as pd
import numpy as np

# Limpar memoria GPU
import torch
import gc

# # 1. Remova as variáveis pesadas (substitua pelos nomes dos seus objetos)
# del model
# del tokenizer
# del model

# 2. Limpe o lixo da memória RAM
gc.collect()

# 3. Esvazie o cache da GPU para o nvidia-smi mostrar como livre
torch.cuda.empty_cache()


# model_name = "Helsinki-NLP/opus-mt-en-ROMANCE"
# tokenizer = MarianTokenizer.from_pretrained(model_name)
# model = MarianMTModel.from_pretrained(model_name).to(device)


In [11]:
# tranformar o arquivo csv complaints_with_sentiment.csv em um um dataframe para análises específicas de reclamações positivas.
df_complaints = pd.read_csv("complaints_with_sentiment.csv")
df_complaints.info()
df_complaints.head()

<class 'pandas.DataFrame'>
RangeIndex: 162421 entries, 0 to 162420
Data columns (total 5 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   product                  162421 non-null  str    
 1   narrative                162411 non-null  str    
 2   clean_narrative          162411 non-null  str    
 3   sentiment_FinancialBERT  162421 non-null  str    
 4   score                    162421 non-null  float64
dtypes: float64(1), str(4)
memory usage: 171.6 MB


,product,narrative,clean_narrative,sentiment_FinancialBERT,score
0,credit_card,purchase order day shipping amount receive pro...,purchas ord day ship amount receiv produc week...,neutral,0.998380
1,credit_card,forwarded message date tue subject please inve...,forward mess dat tue subject pleas investig co...,neutral,0.997814
2,retail_banking,forwarded message cc sent friday pdt subject f...,forward mess cc sent friday pdt subject fin le...,neutral,0.985487
3,credit_reporting,payment history missing credit report speciali...,pay hist miss credit report spec loan serv sl ...,neutral,0.995491
4,credit_reporting,payment history missing credit report made mis...,pay hist miss credit report mad mistak put acc...,neutral,0.995610


In [12]:
# No data frame df_complaints separar os dados segundo a catagoria de sentiment_FinancialBERT iguais a "positive".
df_positive = df_complaints[df_complaints['sentiment_FinancialBERT'] == 'positive']
df_positive.info()
df_positive.head()

<class 'pandas.DataFrame'>
Index: 779 entries, 1779 to 162285
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   product                  779 non-null    str    
 1   narrative                779 non-null    str    
 2   clean_narrative          779 non-null    str    
 3   sentiment_FinancialBERT  779 non-null    str    
 4   score                    779 non-null    float64
dtypes: float64(1), str(4)
memory usage: 425.6 KB


,product,narrative,clean_narrative,sentiment_FinancialBERT,score
1779,credit_reporting,delinquent debt paid since credit bureau updat...,delinqu debt paid sint credit bureau upd recor...,positive,0.903719
1780,credit_reporting,delinquent debt paid since credit bureau updat...,delinqu debt paid sint credit bureau upd recor...,positive,0.903719
2209,credit_card,scam artist created post looked like branch ap...,scam art cre post look lik branch ap invit joi...,positive,0.517733
2586,credit_card,two year ago clear blue sky capital one arbitr...,two year ago clear blu sky capit on arbit deci...,positive,0.631005
3823,retail_banking,account wont link pnc need pnc wont let save e...,account wont link pnc nee pnc wont let sav eith,positive,0.999597


In [13]:
# retira as linhas cujo valor da coluna "narrative" seja igual a NaN.
df_positive = df_positive.dropna(subset=['narrative'])
df_positive.info()
df_positive.head()

<class 'pandas.DataFrame'>
Index: 779 entries, 1779 to 162285
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   product                  779 non-null    str    
 1   narrative                779 non-null    str    
 2   clean_narrative          779 non-null    str    
 3   sentiment_FinancialBERT  779 non-null    str    
 4   score                    779 non-null    float64
dtypes: float64(1), str(4)
memory usage: 425.6 KB


,product,narrative,clean_narrative,sentiment_FinancialBERT,score
1779,credit_reporting,delinquent debt paid since credit bureau updat...,delinqu debt paid sint credit bureau upd recor...,positive,0.903719
1780,credit_reporting,delinquent debt paid since credit bureau updat...,delinqu debt paid sint credit bureau upd recor...,positive,0.903719
2209,credit_card,scam artist created post looked like branch ap...,scam art cre post look lik branch ap invit joi...,positive,0.517733
2586,credit_card,two year ago clear blue sky capital one arbitr...,two year ago clear blu sky capit on arbit deci...,positive,0.631005
3823,retail_banking,account wont link pnc need pnc wont let save e...,account wont link pnc nee pnc wont let sav eith,positive,0.999597


In [14]:
# Verificar se a GPU está disponível
print(f"GPU disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU disponível: True
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [ ]:
# Recomendação: Para sua RTX 3060 6GB, use deepseek-r1:7b com quantização Q4,
#  ou deepseek-r1:8b se disponível. Execute no bash
# ollama pull deepseek-r1:7b

In [ ]:
# installar as bibliotecas necessárias
# pip install ollama pandas tqdm  <<< aqui no VSCode em projeto_tech_05

In [ ]:
# usado no OLLAMA.
texts_positive = df_positive["narrative"].tolist()

In [ ]:
# transforar texts_positive em arquivo csv para análises específicas de reclamações positivas.
df_positive.to_csv("complaints_positive.csv", index=False)


In [18]:
import pandas as pd
import ollama
from tqdm import tqdm
import time

def gerar_variacoes_local(frase_original, num_variacoes=3):
    """
    Gera variações de frases usando DeepSeek rodando localmente.
    """
    prompt = f"""
    Frase original: "{frase_original}"

    Gere {num_variacoes} versões diferentes desta frase, mantendo o mesmo significado,
    em Ingles. Responda apenas com as frases, uma por linha.
    """

    try:
        response = ollama.chat(
            model='deepseek-r1:1.5b',  # ou o modelo que você baixou
            messages=[
                {'role': 'system', 'content': 'Você é um assistente que gera variações de textos.'},
                {'role': 'user', 'content': prompt}
            ],
            options={
                'temperature': 0.8,
                'num_predict': 200
            }
        )

        resultado = response['message']['content'].strip()
        variacoes = [linha.strip() for linha in resultado.split('\n') if linha.strip()]

        # Limpar possíveis marcações como "1.", "2." etc.
        variacoes_limpas = []
        for v in variacoes:
            # Remove numerações iniciais
            import re
            v_limpa = re.sub(r'^\d+[\.\)]\s*', '', v)
            variacoes_limpas.append(v_limpa)

        return variacoes_limpas[:num_variacoes]

    except Exception as e:
        print(f"Erro: {e}")
        return [f"Erro: {frase_original}"] * num_variacoes

In [19]:
# ============================================
# Processamento similar ao anterior
# ============================================

df = pd.read_csv('complaints_positive.csv')

# Identifique sua coluna de texto
coluna_texto = df.columns[1]  # ou especifique pelo nome
print(f"Processando coluna: {coluna_texto}")

novas_frases = []
frases_originais = []

for idx, row in tqdm(df.head(50).iterrows(), total=50, desc="Testando"):  # Comece com 50 para testar
    frase = row[coluna_texto]
    variacoes = gerar_variacoes_local(frase, num_variacoes=3)

    for variacao in variacoes:
        novas_frases.append(variacao)
        frases_originais.append(frase)

df_resultado = pd.DataFrame({
    'frase_original': frases_originais,
    'frase_gerada': novas_frases
})

Processando coluna: narrative


Testando:   0%|          | 0/50 [00:00<?, ?it/s]

Testando: 100%|██████████| 50/50 [06:21<00:00,  7.64s/it]


In [21]:

df_resultado.to_csv('frases_geradas_local.csv', index=False, encoding='utf-8')
print(f"✅ Geradas {len(df_resultado)} frases!")
#Erro: model requires more system memory (5.5 GiB) than is available (2.8 GiB) (status code: 500)

✅ Geradas 4 frases!
